In [1]:
#import libraries
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
import seaborn as sns
import matplotlib.pyplot as plt 
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix


In [2]:
DATA_PATH = Path("../data/raw/dataset.csv")

if not DATA_PATH.exists():
    raise FileNotFoundError(
        "Dataset introuvable. Consultez le README pour le télécharger."
    )

df = pd.read_csv(DATA_PATH)

In [3]:
binary_cols = [
    "gender",
    "Partner",
    "Dependents",
    "PhoneService",
    "PaperlessBilling"
]

ohe_cols = [
    "MultipleLines",
    "InternetService",
    "OnlineSecurity",
    "OnlineBackup",
    "DeviceProtection",
    "TechSupport",
        "StreamingTV",
    "StreamingMovies",
    "Contract",
    "PaymentMethod"
]

encoder = ColumnTransformer(
    transformers=[
        ("binary", OneHotEncoder(drop="if_binary"), binary_cols),
        ("ohe", OneHotEncoder(handle_unknown="ignore"), ohe_cols)
    ],
    remainder="passthrough"
)
# Variable cible
y = df['Churn'].map({'Yes':1, 'No':0})

# Suppression des colonnes inutiles
X = df.drop(['customerID', 'Churn'], axis=1)

# Conversion de TotalCharges
X['TotalCharges'] = pd.to_numeric(X['TotalCharges'], errors='coerce')
X = X.fillna(X.median(numeric_only=True))

# Encodage
X = pd.get_dummies(X, drop_first=True)


In [4]:
# Choix de technologie

Je recommande un modèle de type transformateur pour données tabulaires (par exemple TabTransformer) pour ce cas de churn.

Justification :
- Les données contiennent beaucoup de variables catégorielles et de relations croisées entre features.
- Les transformateurs tabulaires apprennent des embeddings contextuels et captent mieux ces interactions que des modèles linéaires ou un simple MLP.
- Les GAN/VAE sont plutôt adaptés à la génération ou à la représentation non supervisée, pas directement au classement final.
- Les GNN nécessitent une structure de graphe explicite, ce qui ne correspond pas naturellement à un dataset tabulaire classique.
- Le transfer learning peut être pertinent avec un modèle pré-entraîné sur du tabulaire, mais un transformateur tabulaire est une approche de pointe plus directe pour ce problème.

SyntaxError: invalid syntax (3450674297.py, line 3)

In [ ]:
# Implémentation du transformateur
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

x_train, x_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42, stratify=y)
x_train, x_val, y_train, y_val = train_test_split(x_train, y_train, test_size=0.25, random_state=42, stratify=y_train)
# Split des données

#logistic regression
baseline = LogisticRegression(random_state=42)
baseline.fit(x_train, y_train)
y_pred = baseline.predict(x_test)
print(classification_report(y_test, y_pred))
print("Précision:", baseline.score(x_test, y_test))
print("Précision sur le set de validation:", baseline.score(x_val, y_val))
print("Précision sur le set d'entraînement:", baseline.score(x_train, y_train))
print(accuracy_score(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))


              precision    recall  f1-score   support

           0       0.85      0.89      0.87      1035
           1       0.65      0.56      0.60       374

    accuracy                           0.80      1409
   macro avg       0.75      0.73      0.74      1409
weighted avg       0.80      0.80      0.80      1409

Précision: 0.8034066713981547
Précision sur le set de validation: 0.8041163946061036
Précision sur le set d'entraînement: 0.8044970414201184
0.8034066713981547
[[923 112]
 [165 209]]
